# RDP HTTPX — Async Historical Interday Summaries (`asyncio.gather`)

Demonstrates a complete end-to-end request flow against [LSEG Data Platform (RDP)](https://developers.lseg.com/en/api-catalog/refinitiv-data-platform/refinitiv-data-platform-apis) using [`httpx`](https://www.python-httpx.org/) in **asynchronous** mode with a shared `httpx.AsyncClient` and `asyncio.gather()`:

1. Authenticate with OAuth 2.0 Password Grant to obtain a Bearer token
2. Fetch historical interday price summaries (daily OHLCV) for **10 RICs concurrently**, throttled by an `asyncio.Semaphore`
3. Inspect each result individually — a single failure does not cancel the rest

> **Note:** `verify=False` disables TLS certificate verification — intended for local/dev environments only (e.g. behind a ZScaler proxy). Remove it for production use.
>
> Jupyter supports **top-level `await`** natively, so `asyncio.run()` is not needed here.

In [10]:
import asyncio
import os
import time
import httpx
from dotenv import load_dotenv

## Configuration

API endpoint paths and the RIC universe are defined as constants. Credentials (`MACHINEID_RDP`, `PASSWORD_RDP`, `APPKEY_RDP`, `RDP_BASE_URL`) are loaded from `src/.env` via `python-dotenv`.

In [11]:
AUTH_TOKEN_URL = "/auth/oauth2/v1/token"
AUTH_REVOKE_URL = "/auth/oauth2/v1/revoke"
HISTORICAL_INTERDAY_SUMMARIES_URL = "/data/historical-pricing/v1/views/interday-summaries/"
HISTORICAL_RICS = ["AAPL.O","MSFT.O","META.O","NVDA.O","GOOG.O","ORCL.N","IBM.N","PLTR.O","AMZN.O","AVGO.O","TSLA.O","CRM.N","AMD.O","INTC.O","CSCO.O"]  # Fetched concurrently

In [12]:
load_dotenv()

# Read credentials and base URL from environment variables.
machine_id = os.getenv("MACHINEID_RDP")
password = os.getenv("PASSWORD_RDP")
app_key = os.getenv("APPKEY_RDP")
base_url = os.getenv("RDP_BASE_URL")

fields = ["TRDPRC_1", "BID", "ASK"]
start_date = "2025-11-01T00:00:00Z"
end_date = "2026-02-28T23:59:59Z"

## Helper Functions

Three async helper functions used by the main execution block. All accept the shared `httpx.AsyncClient` instance so the same connection pool is reused across every call.

| Function | Purpose |
|---|---|
| `post_authentication` | OAuth 2.0 Password Grant — returns a Bearer token |
| `post_auth_revoke` | Invalidates the session token when done, using HTTP Basic Auth |
| `get_historical_interday_summaries` | Fetches daily OHLCV data for a single RIC; accepts an optional `asyncio.Semaphore` to cap concurrency |

In [13]:
async def post_authentication(machine_id, password, app_key, url, client):
    """Authenticate to RDP and return the token response as JSON."""
    # Build the OAuth 2.0 Password Grant request payload.
    # Sent as application/x-www-form-urlencoded (httpx encodes a dict automatically).
    payload = {
        "username": machine_id,                   # RDP Machine-ID
        "password": password,                     # RDP Password
        "grant_type": "password",                 # OAuth 2.0 grant type
        "scope": "trapi",                         # Target API scope
        "takeExclusiveSignOnControl": "true",     # Revoke other active sessions
        "client_id": app_key                      # RDP AppKey (acts as client_id)
    }

    headers = {
        "Content-Type": "application/x-www-form-urlencoded"
    }

    # Send authentication request to the OAuth token endpoint.
    # `data=payload` sends a form body required by this endpoint.
    response_auth = await client.post(url, data=payload, headers=headers)
    response_auth.raise_for_status()  # Raise for 4xx/5xx API failures.
    return response_auth.json()

In [14]:
async def post_auth_revoke(token, app_key, url, client):
    """Revoke the access token to end the session."""
    headers = {
        "Content-Type": "application/x-www-form-urlencoded"
    }

    payload = f"token={token}"
    auth = httpx.BasicAuth(username=app_key, password="")
    response = await client.post(url, data=payload, headers=headers, auth=auth)
    response.raise_for_status()

In [15]:
async def get_historical_interday_summaries(ric, token, url, client, interval, start, end, fields, semaphore=None):
    """Request historical Interday summaries data for a single RIC."""
    print(f"Fetching historical interday summaries... for RIC: {ric}")

    # Bearer token authenticates the caller; Content-Type signals a JSON response is expected.
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }

    # Build the query string for the interday-summaries endpoint.
    # adjustments: apply standard corporate-action and price corrections so that
    # the returned series is comparable across the full date range.
    # fields: comma-separated list of data columns to include in the response.
    params = {
        "interval": interval,
        "start": start,
        "end": end,
        "adjustments": "exchangeCorrection,manualCorrection,CCH,CRE,RPO,RTS",
        "fields": ",".join(fields)
    }

    # Acquire the semaphore slot before sending the request so that at most
    # `semaphore._value` (e.g. 3) requests are in-flight at the same time.
    # When no semaphore is provided, send the request immediately without throttling.
    if semaphore:
        async with semaphore:
            response_history = await client.get(f"{url}{ric}", params=params, headers=headers)
    else:
        response_history = await client.get(f"{url}{ric}", params=params, headers=headers)

    print(f"Request URL: {response_history.url}")

    # Raise an exception for 4xx/5xx HTTP errors; lets the caller handle
    # status-specific logic (e.g. 429 rate-limit vs. 401 auth failure).
    response_history.raise_for_status()

    # Deserialise and return the JSON payload for further processing by the caller.
    return response_history.json()

## Main Execution

Start the wall-clock timer, then open a single shared `httpx.AsyncClient`. The `async with` block guarantees the client and its connection pool are closed cleanly regardless of success or error.

All RIC requests are dispatched **concurrently** via `asyncio.gather()`. An `asyncio.Semaphore` caps how many requests are in-flight at once (default: 3). `return_exceptions=True` ensures that one failing RIC does not cancel the others — each result is inspected individually after all tasks complete.

In [16]:
start_time = time.perf_counter()

### Open a shared `httpx.AsyncClient`

Using an `async with` block ensures the client's connection pool is closed when the block exits — whether it completes normally or raises an exception. All API calls share the same underlying TCP connections, reducing overhead compared to creating a new client per request.

In [17]:
async with httpx.AsyncClient(
    verify=False,
    base_url=base_url,
    timeout=10.0,
    follow_redirects=True,
) as client:
    # --- Authentication (must complete before any data requests) ---
    try:
        token_data = await post_authentication(machine_id, password, app_key, AUTH_TOKEN_URL, client)
        print("Authentication successful. Access token obtained.")

        access_token = token_data.get("access_token")

        # Limit how many RIC requests run simultaneously to avoid
        # overwhelming the server or hitting rate limits.
        max_concurrent_tasks = 3
        sem = asyncio.Semaphore(max_concurrent_tasks)

        # Build one coroutine per RIC; the semaphore inside each call
        # ensures at most max_concurrent_tasks run at the same time.
        tasks_history = [
            get_historical_interday_summaries(
                ric, access_token, HISTORICAL_INTERDAY_SUMMARIES_URL, client,
                interval="P1D", start=start_date, end=end_date, fields=fields, semaphore=sem
            )
            for ric in HISTORICAL_RICS
        ]

        # gather() runs all tasks concurrently. return_exceptions=True
        # prevents a single failure from cancelling the remaining tasks —
        # each exception is returned as a value instead of being raised.
        results_history = await asyncio.gather(*tasks_history, return_exceptions=True)

        # Pair each RIC with its result (or exception) and handle individually.
        for ric, result in zip(HISTORICAL_RICS, results_history):
            if isinstance(result, httpx.HTTPStatusError):
                raise result   # 4xx / 5xx HTTP response
            elif isinstance(result, httpx.RequestError):
                raise result   # network-level failure (includes timeouts)
            elif isinstance(result, Exception):
                raise result   # any other unexpected error
            print(f"Historical interday summaries for '{ric}': {result}\n\n")

        await asyncio.sleep(1)  # Just to ensure all output is printed before revoking the token.
        print("Revoking access token...")
        await post_auth_revoke(access_token, app_key, AUTH_REVOKE_URL, client)
        print("Access token revoked successfully.\n")

    # --- Exception handlers ordered from most-specific to least-specific ---
    except httpx.HTTPStatusError as e:
        # Server returned a 4xx or 5xx status code.
        print(f"HTTP error during request: {e.request.url} {e.response.status_code} - {e.response.text}")
    except httpx.TimeoutException as e:
        # Request exceeded the configured timeout (must precede RequestError
        # because TimeoutException is a subclass of RequestError).
        print(f"Timeout error: {e}")
    except httpx.RequestError as e:
        # Network-level failure: DNS, connection refused, SSL error, etc.
        print(f"Network error: {e}")
    except Exception as e:
        # Catch-all for unexpected errors (e.g. JSON decode, assertion).
        print(f"Unexpected error: {e}")

Authentication successful. Access token obtained.
Fetching historical interday summaries... for RIC: AAPL.O
Fetching historical interday summaries... for RIC: MSFT.O
Fetching historical interday summaries... for RIC: META.O
Fetching historical interday summaries... for RIC: NVDA.O
Fetching historical interday summaries... for RIC: GOOG.O
Fetching historical interday summaries... for RIC: ORCL.N
Fetching historical interday summaries... for RIC: IBM.N
Fetching historical interday summaries... for RIC: PLTR.O
Fetching historical interday summaries... for RIC: AMZN.O
Fetching historical interday summaries... for RIC: AVGO.O
Fetching historical interday summaries... for RIC: TSLA.O
Fetching historical interday summaries... for RIC: CRM.N
Fetching historical interday summaries... for RIC: AMD.O
Fetching historical interday summaries... for RIC: INTC.O
Fetching historical interday summaries... for RIC: CSCO.O
Request URL: https://api.refinitiv.com/data/historical-pricing/v1/views/interday-su

In [ ]:
elapsed = time.perf_counter() - start_time
print(f"async_call_nb.ipynb executed for {HISTORICAL_RICS} ({len(HISTORICAL_RICS)} RICs) in {elapsed:0.2f} seconds.")

async_call_nb.ipynb executed for ['AAPL.O', 'MSFT.O', 'META.O', 'NVDA.O', 'GOOG.O', 'ORCL.N', 'IBM.N', 'PLTR.O', 'AMZN.O', 'AVGO.O', 'TSLA.O', 'CRM.N', 'AMD.O', 'INTC.O', 'CSCO.O'] in 9.61 seconds.
